# 05 — Ontology-informed machine learning (composite-label binary classification, tuned models)

This notebook evaluates whether an ontology-informed baseline representation improves classification of short-term progression when compared with a conventional raw-variable representation.

**Binary target**:
- `slow`
- `fast`

The binary label is imported from Step 04, where it is derived from a composite progression score based on longitudinal change in:
- `updrs3_score`
- `hy`
- `moca`

**Important anti-leakage rule**:
Because the class label is defined using these three variables, the following baseline predictors are excluded from all feature sets:
- `updrs3_score`
- `hy`
- `moca`

**Inputs**:
- `output/trajectory_analysis/subject_progression_table.csv`
- `output/trajectory_analysis/progression_visit_level_table.csv`
- `mapping/ppmi_pdon_pmdo_mapping.csv`

**Representations evaluated**:
1. **Conventional**: baseline raw variables, excluding `updrs3_score`, `hy`, and `moca`
2. **Ontology-informed**: semantic aggregates and domain-level features, recomputed without `updrs3_score`, `hy`, and `moca`

**Models**:
- Logistic regression
- Decision tree
- Random forest

**Additional procedures**:
1. Grid search for model-specific hyperparameters
2. ROC-AUC evaluation alongside the other reported metrics

**Main outputs**:
- baseline modelling dataset
- feature matrices
- best hyperparameters for each representation/model pair
- cross-validated classification metrics
- ROC curves and confusion matrices
- exploratory feature summaries saved to `output/ontology_informed_ml/`


In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn

## 1) Project root (Drive recommended)

In [ ]:
from pathlib import Path
import os

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/ppmi-ontology-alignment')
else:
    PROJECT_DIR = Path('/content/ppmi-ontology-alignment')

MAP_DIR = PROJECT_DIR / 'mapping'
OUT_DIR = PROJECT_DIR / 'output'
TRAJ_DIR = OUT_DIR / 'trajectory_analysis'
ML_DIR = OUT_DIR / 'ontology_informed_ml'
FIG_DIR = ML_DIR / 'figures'
for p in [MAP_DIR, OUT_DIR, TRAJ_DIR, ML_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MAP_PATH = MAP_DIR / 'ppmi_pdon_pmdo_mapping.csv'
PROG_PATH = TRAJ_DIR / 'subject_progression_table.csv'
VISIT_PATH = TRAJ_DIR / 'progression_visit_level_table.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('MAP_PATH   :', MAP_PATH, 'exists=', MAP_PATH.exists())
print('PROG_PATH  :', PROG_PATH, 'exists=', PROG_PATH.exists())
print('VISIT_PATH :', VISIT_PATH, 'exists=', VISIT_PATH.exists())
print('ML_DIR     :', ML_DIR)

## 2) Load inputs

In [ ]:
import pandas as pd
import numpy as np

if not MAP_PATH.exists():
    raise FileNotFoundError(f'Missing mapping file: {MAP_PATH}')
if not PROG_PATH.exists():
    raise FileNotFoundError(f'Missing progression file: {PROG_PATH}')
if not VISIT_PATH.exists():
    raise FileNotFoundError(f'Missing visit-level file: {VISIT_PATH}')

mapping = pd.read_csv(MAP_PATH, dtype=str).fillna('')
progression = pd.read_csv(PROG_PATH)
visit_level = pd.read_csv(VISIT_PATH)

print('Mapping rows:', len(mapping))
print('Progression rows:', len(progression))
print('Visit-level rows:', len(visit_level))
print('Unique subjects in progression:', progression['PATNO'].astype(str).nunique())
print('Unique subjects in visit-level:', visit_level['PATNO'].astype(str).nunique())
progression.head(3)

## 3) Define baseline modelling dataset

This notebook uses the first visit of the fixed two-visit pair (`V04`) as the predictor time point and the composite progression label from Step 04 as the binary outcome.

In [ ]:
def normalize_event_id(x):
    return str(x).strip().upper()

visit_level['PATNO'] = visit_level['PATNO'].astype(str).str.strip()
visit_level['EVENT_ID_norm'] = (
    visit_level['EVENT_ID_norm'].astype(str).str.strip().str.upper()
    if 'EVENT_ID_norm' in visit_level.columns
    else visit_level['EVENT_ID'].apply(normalize_event_id)
)

progression['PATNO'] = progression['PATNO'].astype(str).str.strip()
progression['progression_group'] = progression['progression_group'].astype(str).str.strip().str.lower()

baseline = visit_level[visit_level['EVENT_ID_norm'] == 'V04'].copy()

for col in ['progression_group', 'annualised_updrs3_change', 'composite_progression_score',
            'progression_group_x', 'progression_group_y',
            'annualised_updrs3_change_x', 'annualised_updrs3_change_y',
            'composite_progression_score_x', 'composite_progression_score_y']:
    if col in baseline.columns:
        baseline = baseline.drop(columns=[col])

merge_cols = ['PATNO', 'progression_group']
for extra_col in ['composite_progression_score', 'annualised_updrs3_change', 'annualised_hy_change', 'annualised_moca_change', 'annualised_moca_worsening']:
    if extra_col in progression.columns:
        merge_cols.append(extra_col)

baseline = baseline.merge(
    progression[merge_cols],
    on='PATNO',
    how='inner'
)

baseline = baseline[baseline['progression_group'].isin(['slow', 'fast'])].copy()
baseline['progression_label'] = baseline['progression_group'].map({'slow': 0, 'fast': 1})

print('Baseline modelling rows:', len(baseline))
print('Unique subjects:', baseline['PATNO'].nunique())
print('Class distribution:')
print(baseline['progression_group'].value_counts())
baseline.head(3)

## 4) Define raw variables and semantic domains

The three variables used to construct the label are removed from all predictor sets:
- `updrs3_score`
- `hy`
- `moca`


In [ ]:
LEAKAGE_VARS = ['updrs3_score', 'hy', 'moca']

SUBJECT_META = ['SEX', 'EDUCYRS', 'fampd_bin', 'APOE']
VISIT_META = ['age_at_visit']
MOTOR_VARS = ['updrs1_score', 'updrs2_score', 'updrs4_score']
COGNITION_VARS = ['cogstate', 'MCI_testscores']
BIOMARKER_VARS = ['abeta', 'ptau', 'asyn']
IMAGING_VARS = [
    'MIA_LOWPUT_EXPECTED',
    'MIA_CAUDATE_L', 'MIA_CAUDATE_R', 'MIA_CAUDATE_BILAT',
    'MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_PUTAMEN_BILAT',
    'MIA_STRIATUM_L', 'MIA_STRIATUM_R', 'MIA_STRIATUM_BILAT'
]

RAW_FEATURES = SUBJECT_META + VISIT_META + MOTOR_VARS + COGNITION_VARS + BIOMARKER_VARS + IMAGING_VARS
RAW_FEATURES = [c for c in RAW_FEATURES if c in baseline.columns and c not in LEAKAGE_VARS]

print('Leakage vars removed:', LEAKAGE_VARS)
print('Predictor features:')
print(RAW_FEATURES)
print('N features:', len(RAW_FEATURES))

## 5) Build conventional feature matrix

In [ ]:
def to_float_safe(x):
    s = str(x).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

X_raw = baseline[['PATNO'] + RAW_FEATURES].copy()
for c in RAW_FEATURES:
    X_raw[c] = X_raw[c].apply(to_float_safe)

if 'APOE' in baseline.columns and 'APOE' in X_raw.columns:
    apoe_raw = baseline['APOE'].astype(str).str.lower().fillna('')
    X_raw['APOE_e4_present'] = apoe_raw.apply(lambda s: 1.0 if 'e4' in s else 0.0)
    X_raw = X_raw.drop(columns=['APOE'])

raw_feature_cols = [c for c in X_raw.columns if c != 'PATNO']
y = baseline['progression_label'].copy()

print('X_raw shape:', X_raw[raw_feature_cols].shape)
print('y shape    :', y.shape)
X_raw.head(3)

## 6) Build ontology-informed feature matrix

The ontology-informed representation uses semantic domain aggregates and region-level summaries, recomputed without the label-defining variables.

In [ ]:
numeric_baseline = baseline.copy()
for c in RAW_FEATURES:
    if c in numeric_baseline.columns:
        numeric_baseline[c] = numeric_baseline[c].apply(to_float_safe)

X_onto = pd.DataFrame({'PATNO': baseline['PATNO'].astype(str).values})

if 'SEX' in baseline.columns:
    X_onto['sex_numeric'] = baseline['SEX'].apply(to_float_safe)
if 'EDUCYRS' in baseline.columns:
    X_onto['education_years'] = baseline['EDUCYRS'].apply(to_float_safe)
if 'fampd_bin' in baseline.columns:
    X_onto['family_history_pd'] = baseline['fampd_bin'].apply(to_float_safe)
if 'age_at_visit' in baseline.columns:
    X_onto['age_at_visit'] = baseline['age_at_visit'].apply(to_float_safe)
if 'APOE' in baseline.columns:
    apoe_raw = baseline['APOE'].astype(str).str.lower().fillna('')
    X_onto['APOE_e4_present'] = apoe_raw.apply(lambda s: 1.0 if 'e4' in s else 0.0)

motor_cols = [c for c in MOTOR_VARS if c in numeric_baseline.columns]
cognition_cols = [c for c in COGNITION_VARS if c in numeric_baseline.columns]
biomarker_cols = [c for c in BIOMARKER_VARS if c in numeric_baseline.columns]
imaging_cols = [c for c in IMAGING_VARS if c in numeric_baseline.columns]

if motor_cols:
    X_onto['motor_mean'] = numeric_baseline[motor_cols].mean(axis=1, skipna=True)
    X_onto['motor_sum'] = numeric_baseline[motor_cols].sum(axis=1, skipna=True)
if cognition_cols:
    X_onto['cognition_mean'] = numeric_baseline[cognition_cols].mean(axis=1, skipna=True)
if biomarker_cols:
    X_onto['biomarker_mean'] = numeric_baseline[biomarker_cols].mean(axis=1, skipna=True)
if imaging_cols:
    X_onto['imaging_mean'] = numeric_baseline[imaging_cols].mean(axis=1, skipna=True)

caudate_cols = [c for c in ['MIA_CAUDATE_L', 'MIA_CAUDATE_R', 'MIA_CAUDATE_BILAT'] if c in numeric_baseline.columns]
putamen_cols = [c for c in ['MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_PUTAMEN_BILAT'] if c in numeric_baseline.columns]
striatum_cols = [c for c in ['MIA_STRIATUM_L', 'MIA_STRIATUM_R', 'MIA_STRIATUM_BILAT'] if c in numeric_baseline.columns]

if caudate_cols:
    X_onto['caudate_mean'] = numeric_baseline[caudate_cols].mean(axis=1, skipna=True)
if putamen_cols:
    X_onto['putamen_mean'] = numeric_baseline[putamen_cols].mean(axis=1, skipna=True)
if striatum_cols:
    X_onto['striatum_mean'] = numeric_baseline[striatum_cols].mean(axis=1, skipna=True)

X_onto['motor_n_obs'] = numeric_baseline[motor_cols].notna().sum(axis=1) if motor_cols else 0
X_onto['cognition_n_obs'] = numeric_baseline[cognition_cols].notna().sum(axis=1) if cognition_cols else 0
X_onto['biomarker_n_obs'] = numeric_baseline[biomarker_cols].notna().sum(axis=1) if biomarker_cols else 0
X_onto['imaging_n_obs'] = numeric_baseline[imaging_cols].notna().sum(axis=1) if imaging_cols else 0

onto_feature_cols = [c for c in X_onto.columns if c != 'PATNO']
print('Ontology-informed features:')
print(onto_feature_cols)
print('N ontology-informed features:', len(onto_feature_cols))
X_onto.head(3)

## 7) Prepare cross-validation and model grids

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

class_counts = y.value_counts()
min_class_size = int(class_counts.min())
n_splits = min(5, min_class_size)
if n_splits < 2:
    raise ValueError('Not enough samples per class for stratified cross-validation.')

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
    'roc_auc': 'roc_auc'
}

models = {
    'logreg': {
        'pipeline': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=4000, random_state=42, class_weight='balanced'))
        ]),
        'param_grid': {
            'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0],
            'clf__solver': ['liblinear', 'lbfgs']
        }
    },
    'decision_tree': {
        'pipeline': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=42))
        ]),
        'param_grid': {
            'clf__criterion': ['gini', 'entropy'],
            'clf__max_depth': [2, 3, 4, 5, None],
            'clf__min_samples_split': [2, 4, 6, 8],
            'clf__min_samples_leaf': [1, 2, 3, 4],
            'clf__ccp_alpha': [0.0, 0.001, 0.005, 0.01]
        }
    },
    'rf': {
        'pipeline': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
        ]),
        'param_grid': {
            'clf__n_estimators': [100, 200, 300],
            'clf__max_depth': [3, 4, 5, None],
            'clf__min_samples_split': [2, 4, 6],
            'clf__min_samples_leaf': [1, 2, 3],
            'clf__max_features': ['sqrt', 'log2', None]
        }
    }
}

print('Cross-validation splits:', n_splits)
print('Class counts:')
print(class_counts)
print('Models:', list(models.keys()))

## 8) Grid search and evaluation for conventional and ontology-informed representations

In [ ]:
results = []
predictions = {}
best_params_rows = []

feature_sets = {
    'conventional': (X_raw[raw_feature_cols], raw_feature_cols),
    'ontology_informed': (X_onto[onto_feature_cols], onto_feature_cols)
}

for rep_name, (Xmat, feature_names) in feature_sets.items():
    for model_name, model_obj in models.items():
        grid = GridSearchCV(
            estimator=model_obj['pipeline'],
            param_grid=model_obj['param_grid'],
            scoring='balanced_accuracy',
            cv=cv,
            n_jobs=-1,
            refit=True
        )
        grid.fit(Xmat, y)
        best_model = grid.best_estimator_

        best_params_row = {
            'representation': rep_name,
            'model': model_name,
            'best_score_balanced_accuracy': float(grid.best_score_)
        }
        best_params_row.update(grid.best_params_)
        best_params_rows.append(best_params_row)

        cv_res = cross_validate(best_model, Xmat, y, cv=cv, scoring=scoring, return_train_score=False)
        y_pred = cross_val_predict(best_model, Xmat, y, cv=cv)
        y_proba = cross_val_predict(best_model, Xmat, y, cv=cv, method='predict_proba')[:, 1]

        predictions[(rep_name, model_name)] = {
            'y_pred': y_pred,
            'y_proba': y_proba,
            'best_model': best_model
        }

        results.append({
            'representation': rep_name,
            'model': model_name,
            'n_features': len(feature_names),
            'cv_accuracy_mean': float(np.mean(cv_res['test_accuracy'])),
            'cv_accuracy_std': float(np.std(cv_res['test_accuracy'])),
            'cv_balanced_accuracy_mean': float(np.mean(cv_res['test_balanced_accuracy'])),
            'cv_balanced_accuracy_std': float(np.std(cv_res['test_balanced_accuracy'])),
            'cv_f1_mean': float(np.mean(cv_res['test_f1'])),
            'cv_f1_std': float(np.std(cv_res['test_f1'])),
            'cv_precision_mean': float(np.mean(cv_res['test_precision'])),
            'cv_precision_std': float(np.std(cv_res['test_precision'])),
            'cv_recall_mean': float(np.mean(cv_res['test_recall'])),
            'cv_recall_std': float(np.std(cv_res['test_recall'])),
            'cv_roc_auc_mean': float(np.mean(cv_res['test_roc_auc'])),
            'cv_roc_auc_std': float(np.std(cv_res['test_roc_auc'])),
            'global_roc_auc': float(roc_auc_score(y, y_proba))
        })

results_df = pd.DataFrame(results).sort_values(['model', 'representation']).reset_index(drop=True)
best_params_df = pd.DataFrame(best_params_rows).sort_values(['model', 'representation']).reset_index(drop=True)

print('--- Best hyperparameters ---')
print(best_params_df)
print('\n--- Cross-validated results ---')
print(results_df)

## 9) Confusion matrices and ROC curves

In [ ]:
import matplotlib.pyplot as plt

label_names = ['slow', 'fast']
cm_rows = []
roc_rows = []

for (rep_name, model_name), pred_obj in predictions.items():
    y_pred = pred_obj['y_pred']
    y_proba = pred_obj['y_proba']

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    cm_rows.append({
        'representation': rep_name,
        'model': model_name,
        'confusion_matrix': cm
    })

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.xticks(range(len(label_names)), label_names)
    plt.yticks(range(len(label_names)), label_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion matrix: {rep_name} / {model_name}')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')
    plt.tight_layout()
    out_path = FIG_DIR / f'confusion_matrix_{rep_name}_{model_name}.png'
    plt.savefig(out_path, dpi=200)
    plt.show()

    fpr, tpr, thresholds = roc_curve(y, y_proba)
    roc_auc = roc_auc_score(y, y_proba)
    roc_rows.append(pd.DataFrame({
        'representation': rep_name,
        'model': model_name,
        'fpr': fpr,
        'tpr': tpr,
        'threshold': thresholds
    }))

    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False positive rate')
    plt.ylabel('True positive rate')
    plt.title(f'ROC curve: {rep_name} / {model_name}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    out_path = FIG_DIR / f'roc_curve_{rep_name}_{model_name}.png'
    plt.savefig(out_path, dpi=200)
    plt.show()

print('Saved confusion matrices and ROC curves to:', FIG_DIR)

## 10) Exploratory feature inspection

These fits are descriptive only and are not out-of-sample estimates.

In [ ]:
feature_catalog = pd.DataFrame(
    [{'representation': 'conventional', 'feature': f} for f in raw_feature_cols] +
    [{'representation': 'ontology_informed', 'feature': f} for f in onto_feature_cols]
)
feature_catalog.head(20)

In [ ]:
feature_importance_rows = []

for rep_name, (Xmat, feature_names) in feature_sets.items():
    for model_name in ['logreg', 'decision_tree', 'rf']:
        best_model = predictions[(rep_name, model_name)]['best_model']
        best_model.fit(Xmat, y)

        if model_name == 'logreg':
            clf = best_model.named_steps['clf']
            importance = np.abs(clf.coef_).reshape(-1)
        else:
            clf = best_model.named_steps['clf']
            importance = clf.feature_importances_

        imp_df = pd.DataFrame({
            'representation': rep_name,
            'model': model_name,
            'feature': feature_names,
            'importance': importance
        }).sort_values('importance', ascending=False)
        feature_importance_rows.append(imp_df)

feature_importance_df = pd.concat(feature_importance_rows, ignore_index=True)
feature_importance_df.head(20)

## 11) Export results

In [ ]:
baseline.to_csv(ML_DIR / 'baseline_modelling_dataset_composite_label.csv', index=False)
X_raw.to_csv(ML_DIR / 'conventional_feature_matrix_composite_label.csv', index=False)
X_onto.to_csv(ML_DIR / 'ontology_informed_feature_matrix_composite_label.csv', index=False)
results_df.to_csv(ML_DIR / 'cross_validated_metrics_composite_label_tuned_models.csv', index=False)
best_params_df.to_csv(ML_DIR / 'best_hyperparameters_composite_label_tuned_models.csv', index=False)
feature_catalog.to_csv(ML_DIR / 'feature_catalog_composite_label.csv', index=False)
feature_importance_df.to_csv(ML_DIR / 'exploratory_feature_importances_composite_label_tuned_models.csv', index=False)

class_summary = pd.DataFrame([
    {'metric': 'n_subjects', 'value': int(len(baseline))},
    {'metric': 'n_slow', 'value': int((baseline['progression_label'] == 0).sum())},
    {'metric': 'n_fast', 'value': int((baseline['progression_label'] == 1).sum())}
])
class_summary.to_csv(ML_DIR / 'class_definition_summary_composite_label.csv', index=False)

cm_export_rows = []
for row in cm_rows:
    cm = row['confusion_matrix']
    for i, true_lab in enumerate(label_names):
        for j, pred_lab in enumerate(label_names):
            cm_export_rows.append({
                'representation': row['representation'],
                'model': row['model'],
                'true_label': true_lab,
                'predicted_label': pred_lab,
                'count': int(cm[i, j])
            })
confusion_export = pd.DataFrame(cm_export_rows)
confusion_export.to_csv(ML_DIR / 'confusion_matrices_composite_label_tuned_models.csv', index=False)

roc_export = pd.concat(roc_rows, ignore_index=True)
roc_export.to_csv(ML_DIR / 'roc_curves_composite_label_tuned_models.csv', index=False)

print('Wrote:')
for fn in [
    'baseline_modelling_dataset_composite_label.csv',
    'conventional_feature_matrix_composite_label.csv',
    'ontology_informed_feature_matrix_composite_label.csv',
    'cross_validated_metrics_composite_label_tuned_models.csv',
    'best_hyperparameters_composite_label_tuned_models.csv',
    'feature_catalog_composite_label.csv',
    'exploratory_feature_importances_composite_label_tuned_models.csv',
    'class_definition_summary_composite_label.csv',
    'confusion_matrices_composite_label_tuned_models.csv',
    'roc_curves_composite_label_tuned_models.csv'
]:
    print('-', ML_DIR / fn)

## 12) Quick interpretation-ready outputs

In [ ]:
print('--- class_definition_summary_composite_label ---')
print(class_summary.to_string(index=False))

print('\n--- best_hyperparameters_composite_label_tuned_models ---')
print(best_params_df.to_string(index=False))

print('\n--- cross_validated_metrics_composite_label_tuned_models ---')
print(results_df.round(3).to_string(index=False))

print('\n--- top exploratory feature importances ---')
for rep in ['conventional', 'ontology_informed']:
    for mdl in ['logreg', 'decision_tree', 'rf']:
        print(f'\n[{rep} / {mdl}]')
        sub = feature_importance_df[(feature_importance_df['representation'] == rep) & (feature_importance_df['model'] == mdl)].head(10)
        print(sub[['feature', 'importance']].round(4).to_string(index=False))